# 0. Why data preprocessing?

## Small idea: preprocessing is a contract

A model needs a stable numerical representation, but research data often contain text,
categories, missing values, repeated participants, and inconsistent measurements.
Preprocessing defines how raw observations become model-ready features **without using
information that would be unavailable for a future observation**.

**Learning goals**

- distinguish raw data, features, targets, identifiers, and audit-only columns;
- separate deterministic rules from transformations that learn from data;
- understand `fit`, `transform`, and data leakage;
- see where preprocessing ends and machine learning begins.

In [ ]:
import numpy as np
import pandas as pd
import sklearn

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## One row, several roles

Suppose each row is one learner response. Columns do not all have the same role:

| Role | Example | Usually sent to a model? |
|---|---|---|
| feature | response length, task type, text | yes, after preprocessing |
| target | `needs_review` | no; it is what we predict |
| identifier | response ID, learner ID | no |
| split group | learner or duplicate cluster | no; used to prevent leakage |
| audit-only | gender, first language | not automatically; often retained for representativeness or fairness checks |

In [ ]:
responses = pd.DataFrame({
    "response_id": ["R01", "R02", "R03", "R04"],
    "learner_id": ["L01", "L01", "L02", "L03"],
    "text": ["من فارسی می‌آموزم", "فارسی زیبا است", "من كتاب دارم", "یادگیری زبان"],
    "task_type": ["free", "picture", "free", "picture"],
    "age": [24, 24, np.nan, 31],
    "first_language": ["Arabic", "Arabic", "Kurdish", "Arabic"],
    "needs_review": [1, 0, 1, 0],
})

target = "needs_review"
features = ["text", "task_type", "age"]
identifiers = ["response_id", "learner_id"]
audit_only = ["first_language"]

assert target not in features
assert set(features).isdisjoint(identifiers + audit_only)
responses

## Deterministic rules versus learned transformations

A **deterministic rule** applies the same fixed logic everywhere. Examples include
harmonizing Arabic `ي/ك` with Persian `ی/ک`, parsing a date, or calculating text length.

A **learned transformation** estimates something from data. Examples include a median,
category vocabulary, scaling mean, standard deviation, or TF–IDF vocabulary. Learned
transformations must be fitted on the training partition only.

In [ ]:
from sklearn.impute import SimpleImputer

train_age = pd.DataFrame({"age": [18.0, 22.0, np.nan, 30.0]})
future_age = pd.DataFrame({"age": [np.nan, 45.0]})

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(train_age)                 # learn from training data
future_age_filled = age_imputer.transform(future_age)  # reuse the same rule

print("Training median:", age_imputer.statistics_[0])
print("Transformed future rows:\n", future_age_filled)

## The central leakage rule

```text
raw data → define prediction task → split rows/groups
         → fit preprocessing on training data
         → transform validation and test data
         → fit and evaluate models in Stage 6
```

Leakage occurs when training features contain information from validation/test rows,
from the target, or from the future. It can make an experiment look excellent while
failing on genuinely unseen data.

In [ ]:
train_tokens = pd.Series([80, 120, 150, np.nan], name="tokens")
test_tokens = pd.Series([2_000, np.nan], name="tokens")

bad_median = pd.concat([train_tokens, test_tokens]).median()
good_median = train_tokens.median()

print("Median learned from all rows (leakage):", bad_median)
print("Median learned from training rows only:", good_median)

## What belongs in this stage?

**Stage 5 covers:** task definition, data audits, splitting, leakage prevention,
missing-value handling, categorical encoding, scaling, transformations, feature
construction, feature selection, pipelines, and text vectorization.

**Stage 6 will cover:** regression and classification algorithms, baselines, fitting,
metrics, cross-validation, hyperparameter search, regularization, and model comparison.

## Tiny checkpoint

For each item, decide whether it is deterministic or learned: Persian character
harmonization, median imputation, one-hot categories, response length, TF–IDF weights,
and a fixed date parser.